# ODS v2: run

Runs the build notebooks in dependency order (Policy → Quote header → the quote products and Interaction in parallel; Reference alongside). Leave `batch_id` blank for a full run. To **resume** a failed run, set `batch_id` to that run's batch id: notebooks that already succeeded in it are skipped.


In [ ]:
%%pyspark
# Parameters
batch_id = ""          # blank = new full run; an earlier batch id = resume it


In [ ]:
%%pyspark
import datetime, json
from pyspark.sql import functions as F
NOTEBOOKS = json.loads(r"""[{"name": "10_Policy", "depends": []}, {"name": "20_Quote", "depends": ["10_Policy"]}, {"name": "21_Quote_Motor", "depends": ["20_Quote"]}, {"name": "22_Quote_Home", "depends": ["20_Quote"]}, {"name": "23_Quote_Travel", "depends": ["20_Quote"]}, {"name": "24_Quote_Van", "depends": ["20_Quote"]}, {"name": "30_Interaction", "depends": ["20_Quote"]}, {"name": "40_Reference", "depends": []}]""")
target_schema = "ods_v2"
import time
batch_id = batch_id or datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d%H%M%S")
# Fabric platform faults (e.g. a 504 while runMultiple fetches the notebooks) are retried; failures inside our
# notebooks are not. Each attempt skips notebooks that already succeeded in this batch, so nothing is redone.
# If runMultiple still fails on a platform fault, the notebooks run one at a time (notebookutils.notebook.run), in
# dependency order: slower, but no bulk notebook fetch.
TRANSIENT = ("Fetch notebook content", "GatewayTimeout", "status code: 504", "timed out", "TooManyRequests", "status code: 429")


def succeeded():
    if not spark.catalog.tableExists(f"{target_schema}.OdsRunLog"):
        return set()
    return {r.Notebook for r in spark.table(f"{target_schema}.OdsRunLog")
                .where((F.col("OdsBatchId") == batch_id) & (F.col("Status") == "succeeded")).select("Notebook").distinct().collect()}


finished = False
for attempt in range(1, 3):
    done = succeeded()
    todo = [n for n in NOTEBOOKS if n["name"] not in done]
    print(f"batch {batch_id}, attempt {attempt}: {'already done: ' + ', '.join(sorted(done)) if done else 'nothing done yet'}")
    if not todo:
        finished = True
        break
    print("to run:", ", ".join(n["name"] for n in todo))
    dag = {"activities": [{"name": n["name"], "path": "ODS_v2_" + n["name"], "timeoutPerCellInSeconds": 7200,
                            "args": {"batch_id": batch_id},
                            "dependencies": [d for d in n["depends"] if d not in done]} for n in todo],
           "concurrency": 4}
    try:
        notebookutils.notebook.runMultiple(dag)
        finished = True
        break
    except Exception as e:  # noqa: BLE001
        msg = str(e)
        if not any(t in msg for t in TRANSIENT):
            raise
        print(f"platform fault on attempt {attempt}: {msg[:300]}")
        if attempt == 1:
            time.sleep(60)
if not finished:
    print("runMultiple kept failing on platform faults: running the notebooks one at a time")
    for n in NOTEBOOKS:
        if n["name"] in succeeded():
            continue
        print("running", n["name"])
        notebookutils.notebook.run("ODS_v2_" + n["name"], 14400, {"batch_id": batch_id})


In [ ]:
%%pyspark
display(spark.table(f"{target_schema}.OdsBuildLog").where(F.col("OdsBatchId") == batch_id).orderBy("TableName"))
